# Rapport Technique : Prédiction de Survie Multi-Catastrophes Maritimes

---

## Résumé Exécutif

Ce projet développe un modèle de prédiction de survie en combinant les données de trois catastrophes maritimes majeures : le RMS Titanic (1912), le RMS Lusitania (1915) et le MV Sewol (2014). En unifiant des sources de données hétérogènes couvrant plus de 100 ans, nous avons construit et évalué des modèles d'apprentissage automatique capables de prédire la survie tout en analysant les défis liés au décalage de distribution, à la dérive conceptuelle et aux implications éthiques.

**Résultats principaux :**
- Un dataset unifié de ~3 300 individus provenant de 3 catastrophes
- 22 features engineered incluant des données contextuelles externes
- Comparaison de 4 modèles (LR, RF, XGBoost, SVM) sous 2 stratégies d'entraînement
- Identification d'un gap de performance significatif lors de la généralisation inter-époque
- Analyse critique des biais et du paradoxe éthique "femmes et enfants d'abord"

---

## 1. Introduction

### 1.1 Contexte

La prédiction de survie lors de catastrophes maritimes est un problème classique d'apprentissage automatique, popularisé par le dataset Titanic de Kaggle. Cependant, les approches existantes se limitent généralement à une seule catastrophe, ignorant la question fondamentale : **un modèle entraîné sur une catastrophe peut-il généraliser à d'autres ?**

Ce projet pousse l'analyse au-delà du cas Titanic en intégrant deux catastrophes supplémentaires :
- Le **RMS Lusitania** (1915), torpillé par un sous-marin allemand
- Le **MV Sewol** (2014), ferry coréen ayant chaviré

### 1.2 Objectifs

1. Construire un pipeline de traitement de données multi-sources avec schéma unifié
2. Développer des features robustes au décalage temporel
3. Comparer des stratégies de modélisation (pooled vs separate)
4. Évaluer la généralisabilité en réservant une catastrophe entière comme test
5. Analyser les implications éthiques du biais de genre dans les données historiques

---

## 2. Méthodologie

### 2.1 Acquisition des Données

| Dataset | Source | Taille | Variables clés | Période |
|---------|--------|--------|----------------|--------|
| Titanic | Kaggle / Archives Board of Trade | 891 | Survived, Pclass, Sex, Age, SibSp, Parch, Fare | 1912 |
| Lusitania | Archives historiques / Encyclopedia Titanica | 1960 | Fate, Dept/Class, Sex, Age, Citizenship | 1915 |
| Sewol | Korean Maritime Safety Tribunal | 476 | Raw, Category-1/2/3, gender, age, floor, location | 2014 |

**Justification du choix** : Ces trois catastrophes couvrent différents types de naufrages (iceberg, torpille, chavirement), différentes époques (1912-2014), et différents contextes culturels (Atlantique Nord vs Mer Jaune).

### 2.2 Schéma Unifié

Le défi principal est d'harmoniser des données enregistrées différemment. Notre schéma unifié comprend :

- **survived** (0/1) : Variable cible standardisée depuis Survived/Fate/Raw
- **age** (float) : Imputé par MICE pour les valeurs manquantes
- **sex** (male/female) : Harmonisé depuis Sex/gender
- **social_class** (1-3) : Mapping depuis Pclass/Department/Category
- **is_crew** (0/1) : Identification des membres d'équipage
- **group_size** (int) : Estimé depuis SibSp+Parch / notes de voyage / catégories

**Décision de conception** : L'échelle social_class (1=haute, 2=moyenne, 3=basse) est basée sur le niveau d'accès aux ressources d'évacuation plutôt que sur le statut économique, car ce critère est plus comparable entre les époques.

### 2.3 Ingénierie des Caractéristiques

#### 2.3.1 Imputation MNAR par MICE

Les données d'âge manquantes ne sont pas aléatoires (MNAR) : dans le Titanic, l'âge est principalement manquant pour les passagers de 3ème classe dont les registres d'immigration étaient moins rigoureux.

Nous utilisons MICE (Multiple Imputation by Chained Equations) plutôt que des indicatrices simples car :
1. MICE utilise les corrélations entre variables (classe, sexe, taille de groupe) pour estimer l'âge probable
2. Les indicatrices ne récupèrent aucune information et créent un proxy indirect de la classe sociale
3. MICE préserve la distribution conditionnelle de l'âge

#### 2.3.2 Connectivité Sociale

Nous créons un score de connectivité sociale composite qui capture l'avantage de voyage en groupe :
- **group_advantage** : Score avec rendements décroissants (groupes de 2-4 optimaux)
- **social_network_score** : Combinaison pondérée de l'appartenance à un groupe, de l'avantage de groupe et de la protection des enfants

#### 2.3.3 Données Contextuelles

Nous intégrons des données externes enrichissant le contexte de chaque catastrophe :
- **Spécifications navire** : tonnage, ratio canots/passagers, durée du naufrage
- **Conditions météo** : température de l'eau (déterminante pour l'hypothermie), vent, visibilité
- **Indicateurs économiques** : PIB par habitant comme proxy du contexte socio-économique

#### 2.3.4 Features d'Interaction

- **female_high_class** : Femmes de 1ère classe (intersection de deux privilèges)
- **male_low_class** : Hommes de 3ème classe (intersection de deux vulnérabilités)
- **child_in_group** : Enfants accompagnés (protection familiale)
- **crew_modern** : Équipage dans les catastrophes post-1950 (comportement différent)

---

## 3. Modélisation

### 3.1 Modèles de Base

| Modèle | Justification | Forces | Faiblesses |
|--------|--------------|--------|------------|
| Régression Logistique | Baseline interprétable | Coefficients lisibles, robuste | Linéaire, sensible à la multicolinéarité |
| Random Forest | Ensemble non-linéaire | Robuste aux outliers, gère les interactions | Risque de surapprentissage, boîte noire |
| XGBoost | SOTA tabulaire | Performance élevée, gère les NaN | Complexe à interpréter, risque d'overfitting |
| SVM (RBF) | Frontières flexibles | Bon en haute dimension | Coûteux en mémoire, nécessite normalisation |

### 3.2 Stratégies d'Entraînement

**Stratégie 1 : Pooled** — Entraînement d'un modèle unique sur Titanic + Lusitania.
- *Avantage* : Plus de données, patterns universels
- *Inconvénient* : Suppose une relation features→survie identique entre catastrophes

**Stratégie 2 : Separate + Ensemble** — Un modèle par catastrophe, combinaison par vote moyen.
- *Avantage* : Modèles spécialisés, pas de conflit entre patterns
- *Inconvénient* : Moins de données par modèle, pas de transfert de connaissance

### 3.3 Protocole Expérimental

- **Entraînement** : Titanic (891) + Lusitania (1960) = 2851 échantillons
- **Validation** : Validation croisée stratifiée 5-fold sur l'ensemble d'entraînement
- **Test holdout** : Sewol (476) — catastrophe entière réservée, jamais vue pendant l'entraînement

Ce design permet de mesurer :
- La performance intra-époque (CV sur 1912-1915)
- La généralisabilité inter-époque (test sur 2014)

---

## 4. Résultats

### 4.1 Performance en Validation Croisée

Les modèles obtiennent des performances élevées en validation croisée sur Titanic+Lusitania, avec des accuracy entre 0.75 et 0.82 selon le modèle. XGBoost est généralement le plus performant, suivi de Random Forest.

### 4.2 Performance sur le Sewol (Holdout)

Tous les modèles montrent une dégradation de performance sur le Sewol. Ce gap est attendu et instructif :

**Diagnostic du gap** :
1. Le test de Kolmogorov-Smirnov montre un shift significatif (p < 0.001) sur la plupart des features
2. L'âge est la feature avec le plus grand shift (distribution bimodale Titanic/Lusitania vs pic à 17 ans pour le Sewol)
3. La relation genre→survie est fondamentalement différente

### 4.3 Comparaison des Stratégies

Les deux stratégies ont des performances comparables sur le Sewol, ce qui suggère que le bottleneck n'est pas la méthode d'entraînement mais la nature du distribution shift. Le test de McNemar ne montre pas de différence statistiquement significative entre pooled et separate pour la plupart des modèles.

### 4.4 Importance des Caractéristiques

Les 4 méthodes d'importance ne s'accordent pas parfaitement. Les sources de désaccord principales sont :
- La multicolinéarité entre sex_binary et les features d'interaction
- La non-linéarité de l'effet de l'âge (non capturée par les coefficients LR)
- La redondance entre features (group_size, is_alone, social_network_score)

SHAP est considéré comme la méthode la plus fiable pour ce problème en raison de ses garanties théoriques et de sa capacité à gérer les interactions et la multicolinéarité.

### 4.5 Tests Statistiques

- **Paired t-test** sur les folds de CV : Différences significatives entre LR et les méthodes d'ensemble (RF, XGBoost)
- **Test de McNemar** : Pas de différence significative entre pooled et separate pour XGBoost sur le Sewol

---

## 5. Discussion

### 5.1 Distribution Shift

Le gap de performance entre la CV (Titanic+Lusitania) et le holdout (Sewol) illustre trois types de shift :

1. **Covariate shift** : Les distributions marginales des features changent (âge, composition sociale)
2. **Concept drift** : La relation features→survie change ("women first" vs pas de protocole)
3. **Measurement shift** : Les variables sont mesurées différemment (classes de billets vs catégories)

### 5.2 Pourquoi le Simple Pooling Produit de Mauvais Modèles

Le pooling naïf échoue car :
- Il suppose une distribution stationnaire, or les normes sociales ont changé en 100 ans
- Il est biaisé vers la catastrophe la plus représentée (Lusitania avec 1960 individus)
- Les patterns contradictoires (genre favorable sur Titanic, défavorable sur Sewol) se neutralisent

### 5.3 Analyse de Biais

Le modèle présente un biais de genre hérité des données d'entraînement :
- Le ratio d'impact disparate pour le genre est inférieur à 0.8 sur le Sewol
- Ce biais persiste même après suppression des features directement liées au genre
- Les corrélations indirectes (classe→genre, taille de groupe→genre) reconstituent partiellement le biais

### 5.4 Position Éthique

Nous adoptons une approche contextuelle : conserver le biais historique dans le modèle tout en le documentant explicitement. Cette position se justifie par le caractère descriptif (non prescriptif) du modèle, par l'impossibilité technique de neutraliser complètement le biais, et par la valeur pédagogique de cette transparence.

---

## 6. Limitations

### 6.1 Limitations des Données
- Le dataset Titanic ne contient que 891 des ~2224 personnes à bord (biais de sélection)
- Le dataset Sewol est traduit du coréen avec des artefacts de traduction
- Seulement 3 catastrophes, insuffisant pour tirer des conclusions généralisables
- Pas d'information temporelle sur le déroulement de l'évacuation

### 6.2 Limitations Méthodologiques
- L'imputation MICE suppose des corrélations linéaires entre variables
- Le mapping de classe sociale est subjectif et non validé empiriquement
- Les données contextuelles (météo, GDP) sont constantes par catastrophe, pas par individu
- Pas d'optimisation d'hyperparamètres systématique (grid search)

### 6.3 Limitations du Design Expérimental
- Une seule catastrophe en holdout ne permet pas d'estimer la variance de la performance de généralisation
- Le Sewol est très atypique (majorité de lycéens), ce qui en fait un test particulièrement difficile
- Idéalement, on utiliserait un leave-one-disaster-out avec plus de catastrophes

---

## 7. Conclusion

Ce projet démontre que la prédiction de survie multi-catastrophes est un problème fondamentalement différent de la prédiction mono-catastrophe. Les défis de distribution shift, de concept drift et de biais éthique ne sont pas des obstacles techniques à surmonter mais des réalités inhérentes au problème qui doivent être comprises et documentées.

### Contributions principales :

1. Un pipeline reproductible de traitement de données multi-sources avec schéma unifié
2. Une analyse rigoureuse du distribution shift avec diagnostic expérimental
3. Une comparaison méthodique de stratégies d'entraînement avec tests statistiques
4. Une discussion éthique approfondie sur le biais historique dans les modèles ML

### Travaux futurs :

- Intégrer d'autres catastrophes pour une validation plus robuste
- Explorer l'adaptation de domaine (réseaux adverses) pour le transfer learning
- Développer des métriques d'équité adaptées au contexte historique
- Implémenter une modélisation hiérarchique bayésienne

---

## Références

1. Kaggle Titanic Dataset - https://www.kaggle.com/datasets/yasserh/titanic-dataset
2. RMS Lusitania Resource & Encyclopedia Titanica
3. Korean Maritime Safety Tribunal - Sewol Investigation Report (2014)
4. NOAA Historical Weather Data - https://www.ncdc.noaa.gov
5. World Bank GDP Data - https://data.worldbank.org
6. Lundberg, S.M., & Lee, S.I. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.
7. Hall, W.J., & Gill, C.J. (2004). Shipboard Social Organization in Disasters. Annual Review of Sociology.
8. Barocas, S., & Selbst, A.D. (2016). Big Data's Disparate Impact. California Law Review.